# Customer Analytics

This notebook demonstrates advanced T-SQL patterns for customer analytics using the AdventureWorksDW2022 data warehouse.

**Questions covered:**
9. Customer Segmentation by Purchase Frequency
10. Customer Lifetime Value (CLV) Ranking
11. First vs. Most Recent Purchase Analysis
12. Customer Retention Cohort Analysis

## Setup

In [ ]:
import urllib
from sqlalchemy import create_engine
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

# Plot settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Database connection - adjust server name as needed
server = 'localhost'
database = 'AdventureWorksDW2022'
conn_str = f'DRIVER={{ODBC Driver 17 for SQL Server}};SERVER={server};DATABASE={database};Trusted_Connection=yes;'
conn_url = f'mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(conn_str)}'
engine = create_engine(conn_url)
print('Connected to AdventureWorksDW2022')

---
## Q9: Customer Segmentation by Purchase Frequency

**Business Question:** Segment customers into quintiles based on their order frequency and calculate average order value per segment.

**Key Techniques:**
- `PERCENT_RANK()` for percentile-based segmentation (preferred over NTILE for skewed distributions)
- Frequency and Monetary value calculations

In [ ]:
q9_sql = """
WITH CustomerMetrics AS (
    SELECT 
        CustomerKey, 
        COUNT(DISTINCT SalesOrderNumber) AS Frequency,
        SUM(SalesAmount) AS Monetary
    FROM dbo.FactInternetSales 
    GROUP BY CustomerKey
),
FrequencyScore AS (
    SELECT 
        CustomerKey,
        Frequency,
        Monetary,
        Monetary / NULLIF(Frequency, 0) AS AvgOrderValue,
        PERCENT_RANK() OVER (ORDER BY Frequency ASC) AS F_Score
    FROM CustomerMetrics
)
SELECT 
    CustomerKey AS CustomerID,
    Frequency AS OrderFrequency,
    Monetary AS TotalRevenue,
    AvgOrderValue,
    F_Score * 100 AS FrequencyPercentile,
    CASE 
        WHEN F_Score >= 0.8 THEN 'Top 20% - Most Frequent'
        WHEN F_Score >= 0.6 THEN 'Next 20% - Frequent'
        WHEN F_Score >= 0.4 THEN 'Middle 20% - Average'
        WHEN F_Score >= 0.2 THEN 'Next 20% - Infrequent'
        ELSE 'Bottom 20% - Least Frequent'
    END AS Segment
FROM FrequencyScore
ORDER BY OrderFrequency DESC
"""

df_q9 = pd.read_sql(q9_sql, conn)
df_q9.head(15)

In [ ]:
# Summary statistics by segment
segment_summary = df_q9.groupby('Segment').agg({
    'CustomerID': 'count',
    'OrderFrequency': 'mean',
    'TotalRevenue': ['mean', 'sum'],
    'AvgOrderValue': 'mean'
}).round(2)
segment_summary.columns = ['Customers', 'Avg Frequency', 'Avg Revenue', 'Total Revenue', 'Avg Order Value']

# Reorder segments
segment_order = ['Top 20% - Most Frequent', 'Next 20% - Frequent', 'Middle 20% - Average', 
                 'Next 20% - Infrequent', 'Bottom 20% - Least Frequent']
segment_summary = segment_summary.reindex(segment_order)
segment_summary

In [ ]:
# Visualization: Customer Segmentation
HIGHLIGHT_COLOR = '#2563eb'
MUTED_COLORS = ['#93c5fd', '#d1d5db', '#e5e7eb', '#f3f4f6']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Color scheme: highlight top segment
colors = [HIGHLIGHT_COLOR] + MUTED_COLORS[:4]

# 1. Customer count by segment
ax1 = axes[0]
segment_summary['Customers'].plot(kind='barh', ax=ax1, color=colors)
ax1.set_xlabel('Number of Customers')
ax1.set_title('Customer Count by Segment', fontweight='bold')
ax1.invert_yaxis()
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# 2. Average Order Value by segment
ax2 = axes[1]
segment_summary['Avg Order Value'].plot(kind='barh', ax=ax2, color=colors)
ax2.set_xlabel('Avg Order Value ($)')
ax2.set_title('Average Order Value by Segment', fontweight='bold')
ax2.invert_yaxis()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# 3. Revenue contribution (pie)
ax3 = axes[2]
ax3.pie(segment_summary['Total Revenue'], labels=segment_summary.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
ax3.set_title('Revenue Contribution by Segment', fontweight='bold')

plt.suptitle('Customer Segmentation by Purchase Frequency', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Q10: Customer Lifetime Value (CLV) Ranking

**Business Question:** Rank customers by total lifetime value within each geographic region using predictive CLV.

**Key Techniques:**
- **Predictive CLV Formula:** `APV x APF x Expected Lifespan`
- Regional churn rate calculation
- `DECLARE` for configurable thresholds
- Multiple CTEs for complex business logic

In [ ]:
q10_sql = """
DECLARE @MaxOrderDate DATE = (SELECT MAX(OrderDate) FROM dbo.FactInternetSales);
DECLARE @ChurnThresholdDays INT = 180;

WITH CustomerMetrics AS (
    SELECT 
        st.SalesTerritoryRegion AS Region,
        c.CustomerKey,
        CONCAT_WS(' ', c.FirstName, c.MiddleName, c.LastName) AS CustomerName,
        SUM(fis.SalesAmount) AS TotalRevenue,
        SUM(fis.SalesAmount - fis.TotalProductCost) AS TotalProfit,
        COUNT(DISTINCT fis.SalesOrderNumber) AS TotalOrders,
        DATEDIFF(DAY, MIN(fis.ORDERDATE), @MaxOrderDate) / 365.25 AS CustomerTenureYears,
        DATEDIFF(DAY, MAX(fis.ORDERDATE), @MaxOrderDate) AS DaysSinceLastOrder
    FROM dbo.FactInternetSales fis
    INNER JOIN dbo.DimSalesTerritory st ON fis.SalesTerritoryKey = st.SalesTerritoryKey
    INNER JOIN dbo.DimCustomer c ON fis.CustomerKey = c.CustomerKey
    GROUP BY st.SalesTerritoryRegion, c.CustomerKey, c.FirstName, c.MiddleName, c.LastName
),
RegionalBenchmarks AS (
    SELECT
        Region,
        CustomerKey,
        CustomerName,
        TotalRevenue,
        TotalProfit,
        TotalOrders,
        CustomerTenureYears,
        DaysSinceLastOrder,
        CASE WHEN DaysSinceLastOrder > @ChurnThresholdDays THEN 1 ELSE 0 END AS IsChurned,
        COUNT(*) OVER (PARTITION BY Region) AS RegionCustomerCount
    FROM CustomerMetrics
),
RegionalChurn AS (
    SELECT
        Region,
        CustomerKey,
        CustomerName,
        TotalRevenue,
        TotalProfit,
        TotalOrders,
        CustomerTenureYears,
        DaysSinceLastOrder,
        IsChurned,
        RegionCustomerCount,
        CAST(SUM(IsChurned) OVER (PARTITION BY Region) AS FLOAT) / RegionCustomerCount AS RegionalChurnRate
    FROM RegionalBenchmarks
), 
PredictiveCLV AS (
    SELECT
        Region,
        CustomerKey,
        CustomerName,
        TotalRevenue,
        TotalProfit,
        TotalOrders,
        CustomerTenureYears,
        DaysSinceLastOrder,
        IsChurned,
        RegionalChurnRate,
        TotalProfit / NULLIF(TotalOrders, 0) AS AvgPurchaseValue,
        TotalOrders / NULLIF(CustomerTenureYears, 0) AS AvgPurchaseFrequency,
        ROUND(1.0 / NULLIF(RegionalChurnRate, 0), 2) AS ExpectedLifespanYears
    FROM RegionalChurn
)
SELECT
    Region,
    CustomerKey,
    CustomerName,
    TotalRevenue,
    TotalProfit,
    TotalOrders,
    CustomerTenureYears,
    DaysSinceLastOrder,
    IsChurned,
    RegionalChurnRate,
    AvgPurchaseValue,
    AvgPurchaseFrequency,
    ExpectedLifespanYears,
    ROUND(AvgPurchaseValue * AvgPurchaseFrequency * ExpectedLifespanYears, 2) AS PredictedCLV,
    RANK() OVER (PARTITION BY Region ORDER BY (AvgPurchaseValue * AvgPurchaseFrequency * ExpectedLifespanYears) DESC) AS CLVRank
FROM PredictiveCLV
ORDER BY Region, CLVRank
"""

df_q10 = pd.read_sql(q10_sql, conn)
df_q10.head(15)

In [ ]:
# Regional summary
regional_summary = df_q10.groupby('Region').agg({
    'CustomerKey': 'count',
    'RegionalChurnRate': 'first',
    'ExpectedLifespanYears': 'first',
    'PredictedCLV': ['mean', 'max'],
    'TotalRevenue': 'sum'
}).round(2)
regional_summary.columns = ['Customers', 'Churn Rate', 'Exp. Lifespan (Yrs)', 'Avg CLV', 'Max CLV', 'Total Revenue']
regional_summary

In [ ]:
# Visualization: CLV Distribution by Region
HIGHLIGHT_COLOR = '#2563eb'
MUTED_COLOR = '#d1d5db'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Box plot of CLV by Region
ax1 = axes[0]
df_q10_filtered = df_q10[df_q10['PredictedCLV'].notna() & (df_q10['PredictedCLV'] > 0)]

# Create box plot with professional colors
bp = df_q10_filtered.boxplot(column='PredictedCLV', by='Region', ax=ax1, patch_artist=True,
                              return_type='dict')
for patch in bp['PredictedCLV']['boxes']:
    patch.set_facecolor(HIGHLIGHT_COLOR)
    patch.set_alpha(0.7)

ax1.set_xlabel('Region')
ax1.set_ylabel('Predicted CLV ($)')
ax1.set_title('CLV Distribution by Region', fontweight='bold')
plt.suptitle('')
ax1.tick_params(axis='x', rotation=45)

# 2. Churn Rate vs Expected Lifespan
ax2 = axes[1]
bars = ax2.bar(regional_summary.index, regional_summary['Churn Rate'] * 100, color=MUTED_COLOR, label='Churn Rate %')
ax2_twin = ax2.twinx()
ax2_twin.plot(regional_summary.index, regional_summary['Exp. Lifespan (Yrs)'], 'o-', 
              linewidth=2.5, markersize=8, color=HIGHLIGHT_COLOR, label='Expected Lifespan')
ax2.set_xlabel('Region')
ax2.set_ylabel('Churn Rate (%)', color='#6b7280')
ax2_twin.set_ylabel('Expected Lifespan (Years)', color=HIGHLIGHT_COLOR)
ax2.set_title('Regional Churn Rate vs Expected Customer Lifespan', fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.spines['top'].set_visible(False)

plt.tight_layout()
plt.show()

---
## Q11: First vs. Most Recent Purchase Analysis

**Business Question:** For each customer, compare their first purchase amount with their most recent purchase.

**Key Techniques:**
- `FIRST_VALUE()` with default frame
- `LAST_VALUE()` with explicit `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`
- Understanding window frame boundaries (critical for LAST_VALUE)

In [ ]:
q11_sql = """
WITH CustomerPurchases AS (
    SELECT 
        CustomerKey,
        OrderDate,
        SalesOrderLineNumber,
        SalesAmount,
        FIRST_VALUE(SalesAmount) OVER (
            PARTITION BY CustomerKey 
            ORDER BY OrderDate, SalesOrderLineNumber
        ) AS FirstPurchaseAmount,
        LAST_VALUE(SalesAmount) OVER (
            PARTITION BY CustomerKey 
            ORDER BY OrderDate, SalesOrderLineNumber
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS MostRecentPurchaseAmount
    FROM dbo.FactInternetSales
)
SELECT DISTINCT
    CustomerKey,
    FirstPurchaseAmount,
    MostRecentPurchaseAmount,
    MostRecentPurchaseAmount - FirstPurchaseAmount AS PurchaseAmountChange,
    CASE 
        WHEN MostRecentPurchaseAmount > FirstPurchaseAmount THEN 'Increased'
        WHEN MostRecentPurchaseAmount < FirstPurchaseAmount THEN 'Decreased'
        ELSE 'No Change'
    END AS Trend
FROM CustomerPurchases
ORDER BY PurchaseAmountChange DESC
"""

df_q11 = pd.read_sql(q11_sql, conn)
df_q11.head(15)

In [ ]:
# Summary statistics
trend_summary = df_q11['Trend'].value_counts()
change_stats = df_q11['PurchaseAmountChange'].describe()

print("Purchase Trend Distribution:")
print(trend_summary)
print("\nPurchase Amount Change Statistics:")
print(change_stats)

In [ ]:
# Visualization: First vs Last Purchase
HIGHLIGHT_COLOR = '#2563eb'
MUTED_COLOR = '#d1d5db'

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Professional colors for trends
trend_colors = {'Increased': HIGHLIGHT_COLOR, 'Decreased': '#ef4444', 'No Change': MUTED_COLOR}

# 1. Trend distribution (pie)
ax1 = axes[0]
pie_colors = [trend_colors[t] for t in trend_summary.index]
ax1.pie(trend_summary, labels=trend_summary.index, autopct='%1.1f%%', 
        colors=pie_colors, startangle=90)
ax1.set_title('Customer Purchase Trend\n(First vs Most Recent)', fontweight='bold')

# 2. Distribution of change amounts
ax2 = axes[1]
df_q11['PurchaseAmountChange'].hist(bins=50, ax=ax2, color=HIGHLIGHT_COLOR, edgecolor='white', alpha=0.8)
ax2.axvline(x=0, color='#ef4444', linestyle='--', linewidth=2)
ax2.set_xlabel('Purchase Amount Change ($)')
ax2.set_ylabel('Number of Customers')
ax2.set_title('Distribution of Purchase Amount Changes', fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# 3. Scatter: First vs Last
ax3 = axes[2]
sample = df_q11.sample(min(1000, len(df_q11)), random_state=42)
color_map = sample['Trend'].map(trend_colors)
ax3.scatter(sample['FirstPurchaseAmount'], sample['MostRecentPurchaseAmount'], 
            c=color_map, alpha=0.5, s=20)
max_val = max(sample['FirstPurchaseAmount'].max(), sample['MostRecentPurchaseAmount'].max())
ax3.plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='No Change Line')
ax3.set_xlabel('First Purchase Amount ($)')
ax3.set_ylabel('Most Recent Purchase Amount ($)')
ax3.set_title('First vs Most Recent Purchase (Sample)', fontweight='bold')
ax3.legend()
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## Q12: Customer Retention Cohort Analysis

**Business Question:** Calculate customer retention rates by cohort month (retention triangle).

**Key Techniques:**
- Cohort assignment using `MIN(OrderDate)` as customer "birth"
- `DATETRUNC()` for month-level cohort grouping
- `DATEDIFF(MONTH, ...)` for cohort aging
- `FIRST_VALUE()` for Month 0 baseline
- `PIVOT` for retention triangle format

In [ ]:
q12_sql = """
WITH CustomerFirstPurchase AS (
    SELECT 
        CustomerKey,
        MIN(OrderDate) AS FirstPurchaseDate
    FROM dbo.FactInternetSales
    GROUP BY CustomerKey
),
CustomerCohorts AS (
    SELECT 
        CustomerKey,
        FirstPurchaseDate,
        DATETRUNC(MONTH, FirstPurchaseDate) AS CohortMonth,
        FORMAT(FirstPurchaseDate, 'MMM-yyyy') AS CohortMonthName
    FROM CustomerFirstPurchase
),
RetentionActivity AS (
    SELECT
        cc.CohortMonth,
        cc.CohortMonthName,
        DATEDIFF(MONTH, cc.CohortMonth, fis.OrderDate) AS MonthNumber,
        COUNT(DISTINCT cc.CustomerKey) AS ActiveCustomers
    FROM CustomerCohorts cc
    INNER JOIN dbo.FactInternetSales fis ON cc.CustomerKey = fis.CustomerKey
    GROUP BY cc.CohortMonth, cc.CohortMonthName, DATEDIFF(MONTH, cc.CohortMonth, fis.OrderDate)
)
SELECT 
    CohortMonth,
    CohortMonthName,
    MonthNumber,
    ActiveCustomers,
    FIRST_VALUE(ActiveCustomers) OVER (PARTITION BY CohortMonth ORDER BY MonthNumber) AS CohortSize,
    CAST(ActiveCustomers AS FLOAT) / FIRST_VALUE(ActiveCustomers) OVER (PARTITION BY CohortMonth ORDER BY MonthNumber) * 100 AS RetentionRate
FROM RetentionActivity
WHERE MonthNumber <= 24
ORDER BY CohortMonth, MonthNumber
"""

df_q12 = pd.read_sql(q12_sql, conn)
df_q12.head(20)

In [ ]:
# Create retention triangle (pivot table)
retention_pivot = df_q12.pivot(index='CohortMonthName', columns='MonthNumber', values='RetentionRate')

# Reorder by actual cohort date
cohort_order = df_q12.drop_duplicates('CohortMonthName').sort_values('CohortMonth')['CohortMonthName'].tolist()
retention_pivot = retention_pivot.reindex(cohort_order)

# Rename columns
retention_pivot.columns = [f'M{i}' for i in retention_pivot.columns]
retention_pivot

In [ ]:
# Visualization: Retention Heatmap
fig, ax = plt.subplots(figsize=(16, 10))

# Professional blue colormap
cmap = sns.light_palette('#2563eb', as_cmap=True)
sns.heatmap(retention_pivot, annot=True, fmt='.0f', cmap=cmap, 
            vmin=0, vmax=100, ax=ax, cbar_kws={'label': 'Retention Rate %'},
            annot_kws={'size': 8}, linewidths=0.5, linecolor='white')

ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Cohort (First Purchase Month)')
ax.set_title('Customer Retention Cohort Analysis (Retention Triangle)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Average retention curve across all cohorts
HIGHLIGHT_COLOR = '#2563eb'

avg_retention = df_q12.groupby('MonthNumber')['RetentionRate'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(avg_retention.index, avg_retention.values, '-o', linewidth=2.5, markersize=6, color=HIGHLIGHT_COLOR)
ax.fill_between(avg_retention.index, avg_retention.values, alpha=0.2, color=HIGHLIGHT_COLOR)

ax.set_xlabel('Months Since First Purchase')
ax.set_ylabel('Average Retention Rate (%)')
ax.set_title('Average Customer Retention Curve (All Cohorts)', fontweight='bold')
ax.set_xticks(range(0, 25, 3))
ax.set_ylim(0, 105)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add annotations for key months
for month in [0, 3, 6, 12, 24]:
    if month in avg_retention.index:
        ax.annotate(f'{avg_retention[month]:.1f}%', 
                    xy=(month, avg_retention[month]), 
                    xytext=(month, avg_retention[month] + 5),
                    ha='center', fontsize=9, fontweight='bold', color=HIGHLIGHT_COLOR)

plt.tight_layout()
plt.show()

---
## Cleanup

In [ ]:
conn.close()
print('Connection closed')